In [1]:
from data_loading.models_dataset import ArchiMateDataset, EcoreDataset


reload_db = True

config_params = dict(
    min_enr = 1.2,
    min_edges = 10,
)
ecore = EcoreDataset('ecore_555', reload=reload_db, **config_params)
modelset = EcoreDataset('modelset', reload=reload_db, remove_duplicates=True, **config_params)
mar = EcoreDataset('mar-ecore-github', reload=reload_db, **config_params)
eamodelset = ArchiMateDataset('eamodelset', reload=reload_db, **config_params)

/home/sali/CMAI-Projects/cm2ml-encodings-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset exists: True, reload: True


Loading Ecore_555: 100%|██████████| 548/548 [00:01<00:00, 452.70it/s]


Loaded Total ecore_555 with 548 graphs
Filtering...
Saving ecore_555 to pickle
Saved ecore_555 to pickle
Loaded ecore_555 with 354 graphs
Dataset exists: True, reload: True


Loading Modelset: 100%|██████████| 4127/4127 [00:03<00:00, 1361.63it/s]


Loaded Total modelset with 2043 graphs
Filtering...
Saving modelset to pickle
Saved modelset to pickle
Loaded modelset with 845 graphs
Dataset exists: True, reload: True


Loading Mar-Ecore-Github: 100%|██████████| 18110/18110 [00:31<00:00, 574.85it/s] 


Loaded Total mar-ecore-github with 18110 graphs
Filtering...
Saving mar-ecore-github to pickle
Saved mar-ecore-github to pickle
Loaded mar-ecore-github with 7036 graphs


Loading Eamodelset: 100%|██████████| 979/979 [00:03<00:00, 272.07it/s]


Total graphs: 969
Saving eamodelset to pickle
Saved eamodelset to pickle
Loaded eamodelset with 456 graphs
Graphs: 456


In [6]:
g = min([g for g in ecore.graphs if g is not None], key=lambda g: g.number_of_edges())

In [10]:
g.nodes(data=True)

NodeDataView({'Scheme': {'name': 'Scheme', 'attributes': ['name'], 'abstract': False}, 'Table': {'name': 'Table', 'attributes': ['name'], 'abstract': False}, 'Column': {'name': 'Column', 'attributes': ['name'], 'abstract': False}, 'FKey': {'name': 'FKey', 'attributes': [], 'abstract': False}, 'PKey': {'name': 'PKey', 'attributes': [], 'abstract': False}})

In [11]:
g.edges(data=True)

OutEdgeDataView([('Scheme', 'Table', {'name': 'tables', 'type': 'containment'}), ('Scheme', 'FKey', {'name': 'keys', 'type': 'containment'}), ('Table', 'Column', {'name': 'columns', 'type': 'containment'}), ('Table', 'Scheme', {'name': 'scheme', 'type': 'reference'}), ('Table', 'PKey', {'name': 'key', 'type': 'containment'}), ('Column', 'Table', {'name': 'table', 'type': 'reference'}), ('FKey', 'PKey', {'name': 'refersTo', 'type': 'reference'}), ('FKey', 'Column', {'name': 'column', 'type': 'reference'}), ('FKey', 'Scheme', {'name': 'scheme', 'type': 'reference'}), ('PKey', 'Column', {'name': 'column', 'type': 'reference'})])

In [2]:
import pandas as pd

df = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_layer/flat_results.csv")

In [3]:
df.drop(columns=['dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr', 'use_attributes', 'use_edge_label', 'graph_encoder_family', 'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'], inplace=True)

In [4]:
df.columns

Index(['encoder_key', 'classification_mode', 'tree', 'path_depth',
       'use_edge_type', 'gnn_model_name', 'accuracy', 'precision_macro',
       'recall_macro', 'f1_macro'],
      dtype='str')

In [19]:
import pandas as pd

encoders = ['bow', 'tfidf', 'w2v', 'bert-encoder']
cls_mode = ['gnn', 'text']
booleans = ["tree", "use_edge_type"]
gnn_models = ['gcn', 'gat', 'sage']

metric_columns = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
parameter_columns = ['encoder_key', 'classification_mode', 'tree', 'use_edge_type', 'gnn_model_name']


def get_unique_values(df, parameter):
    if parameter == "encoder_key":
        return encoders
    elif parameter == "classification_mode":
        return cls_mode
    elif parameter in ["tree", "use_edge_type"]:
        return [False, True]
    elif parameter == "gnn_model_name":
        return gnn_models
    else:
        raise ValueError(f"Unknown parameter: {parameter}")


def get_transition_groups(df, parameter_of_interest):
    """
    Returns a list of transition reports where the parameter_of_interest changes
    across consecutive values while all other parameters remain fixed.

    Output format:
    [
        {
            "transition": ("bow", "tfidf"),
            "fixed_params": {
                "classification_mode": "gnn",
                "tree": False,
                "use_edge_type": True,
                "gnn_model_name": "gcn"
            },
            "delta_report": {
                "before": {"accuracy": 0.1, ...},
                "after": {"accuracy": 0.15, ...},
                "delta": {"accuracy": 0.05, ...}
            }
        },
        ...
    ]
    """
    unique_values = get_unique_values(df, parameter_of_interest)
    print(f"Unique values for {parameter_of_interest}: {unique_values}")
    transitions = list(zip(unique_values[:-1], unique_values[1:]))

    fixed_parameters = [p for p in parameter_columns if p != parameter_of_interest]
    results = []

    # group by all parameters except the one that changes
    grouped = df.groupby(fixed_parameters, dropna=False)

    for fixed_key, group in grouped:
        if len(fixed_parameters) == 1:
            fixed_key = (fixed_key,)

        fixed_params = dict(zip(fixed_parameters, fixed_key))

        # Build lookup from parameter_of_interest value -> row
        # Assumes exactly one row per config; if multiple exist, aggregate first.
        value_to_row = {}
        for _, row in group.iterrows():
            value = row[parameter_of_interest]
            value_to_row[value] = row

        for before_val, after_val in transitions:
            if before_val in value_to_row and after_val in value_to_row:
                before_row = value_to_row[before_val]
                after_row = value_to_row[after_val]

                before_metrics = {
                    metric: float(before_row[metric]) for metric in metric_columns
                }
                after_metrics = {
                    metric: float(after_row[metric]) for metric in metric_columns
                }
                delta_metrics = {
                    metric: after_metrics[metric] - before_metrics[metric]
                    for metric in metric_columns
                }

                results.append({
                    "transition": (before_val, after_val),
                    "fixed_params": fixed_params,
                    "delta_report": {
                        "before": before_metrics,
                        "after": after_metrics,
                        "delta": delta_metrics
                    }
                })

    return results

def filter_transition_groups(reports, filter_params):
    """
    Filters the transition reports based on specified fixed parameter values.

    filter_params example:
    {
        "classification_mode": "gnn",
        "tree": False,
        "use_edge_type": True,
        "gnn_model_name": "gcn"
    }
    """
    filtered = []
    for report in reports:
        match = True
        for param, value in filter_params.items():
            if report["fixed_params"].get(param) != value:
                match = False
                break
        if match:
            filtered.append(report)
    return filtered

In [20]:
reports = get_transition_groups(df, "classification_mode")
print(f"Total transitions found: {len(reports)}")
for r in reports[:3]:
    print(r)

Unique values for classification_mode: ['gnn', 'text']


KeyboardInterrupt: 

In [13]:
filtered_reports = filter_transition_groups(reports, {
    "classification_mode": "gnn",
    "tree": False,
    "use_edge_type": True,
    "gnn_model_name": "gcn"
})

In [14]:
for r in filtered_reports:
    print(r)

{'transition': ('bow', 'tfidf'), 'fixed_params': {'classification_mode': 'gnn', 'tree': False, 'use_edge_type': True, 'gnn_model_name': 'gcn'}, 'delta_report': {'before': {'accuracy': 0.783248730964467, 'precision_macro': 0.7661840658844498, 'recall_macro': 0.5886054719372746, 'f1_macro': 0.641278143941878}, 'after': {'accuracy': 0.7736040609137056, 'precision_macro': 0.7670827697354452, 'recall_macro': 0.5537175758755932, 'f1_macro': 0.6047266031070722}, 'delta': {'accuracy': -0.009644670050761417, 'precision_macro': 0.0008987038509954415, 'recall_macro': -0.03488789606168141, 'f1_macro': -0.03655154083480583}}}
{'transition': ('tfidf', 'w2v'), 'fixed_params': {'classification_mode': 'gnn', 'tree': False, 'use_edge_type': True, 'gnn_model_name': 'gcn'}, 'delta_report': {'before': {'accuracy': 0.7736040609137056, 'precision_macro': 0.7670827697354452, 'recall_macro': 0.5537175758755932, 'f1_macro': 0.6047266031070722}, 'after': {'accuracy': 0.5707275803722505, 'precision_macro': 0.5910